앞서 2장에 걸쳐 소개한 `@property`의 가장 큰 단점은 재사용성이다.

아래는 학생의 숙제 점수를 부여하는 예제이다. `@property`를 이용해서 쉽게 구현이 가능하다.

In [ ]:
class Homework:
    def __init__(self):
        self._grade = 0

    @property
    def grade(self):
        return self._grade

    @grade.setter
    def grade(self, value):
        if not (0 <= value <= 100):
            raise ValueError(
                '점수는 0과 100 사이입니다')
        self._grade = value

galileo = Homework()
galileo.grade = 95

이번에는 학생에게 시험 점수를 부여하는 예제이다. 시험 과목마다 새로운 `@property`를 지정하는 반복이 생긴다.

In [ ]:
class Exam:
    def __init__(self):
        self._writing_grade = 0
        self._math_grade = 0

    @staticmethod
    def _check_grade(value):
        if not (0 <= value <= 100):
            raise ValueError(
                '점수는 0과 100 사이입니다')

    @property
    def writing_grade(self):
        return self._writing_grade

    @writing_grade.setter
    def writing_grade(self, value):
        self._check_grade(value)
        self._writing_grade = value

    @property
    def math_grade(self):
        return self._math_grade

    @math_grade.setter
    def math_grade(self, value):
        self._check_grade(value)
        self._math_grade = value

이걸 해결하기 위해서 디스크립터를 사용한다. 디스크립터 프로토콜은 파이썬에서 애트리뷰트 접근을 해석하는 방법을 정의한다.

`__get__`, `__set__` 메서드를 제공하고, 이걸 활용하면 별다른 준비없이 원하는 구현이 가능하다. 이게 동작하는 원리는 아래와 같다.
```python
exam = Exam()
exam.writing_grade = 40
--> Exam.__dict__['writing_grade'].__set__(exam, 40)

exam.writing_grade
--> Exam.__dict__['writing_grade'].__get__(exam, Exam)
```

In [ ]:
class Grade:
    def __init__(self):
        self._value = 0
        
    def __get__(self, instance, instance_type):
        return self._value
    
    def __set__(self, instance, value):
        if not (0 <= value <= 100):
            raise ValueError(
                '점수는 0과 100 사이입니다')
        self._value = value
        
class Exam:
    # 클래스 애트리뷰트
    math_grade = Grade()
    writing_grade = Grade()
    science_grade = Grade()

그런데 원하는대로 동작하지 않는다.

In [ ]:
first_exam = Exam()
first_exam.writing_grade = 82
first_exam.science_grade = 99
print('쓰기', first_exam.writing_grade)
print('과학', first_exam.science_grade)

second_exam = Exam()
second_exam.writing_grade = 75
print(f'두 번째 쓰기 점수 {second_exam.writing_grade} 맞음')
print(f'첫 번째 쓰기 점수 {first_exam.writing_grade} 틀림; 82점이어야 함')

`Grade` 인스턴스를 모든 `Exam` 인스턴스가 공유해서 발생한 문제이다. 인스턴스별 상태를 딕셔너리에 저장하면 해결 가능하다.

In [ ]:
class Grade:
    def __init__(self):
        self._values = {}

    def __get__(self, instance, instance_type):
        if instance is None:
            return self
        return self._values.get(instance, 0)

    def __set__(self, instance, value):
        if not (0 <= value <= 100):
            raise ValueError(
                '점수는 0과 100 사이입니다')
        self._values[instance] = value


class Exam:
    # 클래스 애트리뷰트
    math_grade = Grade()
    writing_grade = Grade()
    science_grade = Grade()

first_exam = Exam()
first_exam.writing_grade = 82
first_exam.science_grade = 99
print('쓰기', first_exam.writing_grade)
print('과학', first_exam.science_grade)

second_exam = Exam()
second_exam.writing_grade = 75
print(f'두번째 쓰기 점수 {second_exam.writing_grade} 맞음')
print(f'첫번째 쓰기 점수 {first_exam.writing_grade} 맞음')

위 방식은 메모리 누수 문제가 있다. `_values`에는 모든 `__set__` 호출로 전달된 `Exam`의 참조를 가지고 있어, 참조 카운터가 `0`이 되지 못하고 _garbage collector_ 에 의해 수집되지 않는다.

이를 해결하기 위해서 약한 참조를 사용한다. `WeakKeyDictionary`라는 특별한 클래스를 사용하면 된다.

In [ ]:
from weakref import WeakKeyDictionary

class Grade:
    def __init__(self):
        self._values = WeakKeyDictionary()

    def __get__(self, instance, instance_type):
        if instance is None:
            return self
        return self._values.get(instance, 0)

    def __set__(self, instance, value):
        if not (0 <= value <= 100):
            raise ValueError(
                '점수는 0과 100 사이입니다')
        self._values[instance] = value


class Exam:
    # 클래스 애트리뷰트
    math_grade = Grade()
    writing_grade = Grade()
    science_grade = Grade()

first_exam = Exam()
first_exam.writing_grade = 82
second_exam = Exam()
second_exam.writing_grade = 75
print(f'첫 번째 쓰기 점수 {first_exam.writing_grade} 맞음')
print(f'두 번째 쓰기 점수 {second_exam.writing_grade} 맞음')